<a href="https://colab.research.google.com/github/lenmecc/miniature-enigma/blob/main/Unificado_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pulp gurobipy openpyxl deap pandas numpy plotly

# %% [1] ----------------------------------------------------------------------
# IMPORTS, CONFIG DEAP Y PARÁMETROS
# -----------------------------------------------------------------------------
import sys
import random
import concurrent.futures
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import pulp
from deap import base, creator, tools, algorithms
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# DEAP global (guardado para no re-crear al re-ejecutar la celda ni chocar en hilos)
if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))  # minimizar Z
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMin)

# --- Parámetros estratégicos ---
PESO_URGENCIA = 80
PESO_DENSIDAD = 20
UMBRAL_RECHAZO = 40.0          # filtro contra WIP inútil
HORAS_TURNO = 8.0              # turno para la evaluación comparativa
LISTA_MAQUINAS = ['LASER 01', 'LASER 02', 'LASER 03', 'LASER 04', 'LASER 05', 'LASER 06', 'LASER 08']

# --- Urgencia graduada para pedidos VENCIDOS (antes: 2.0 plano para todos) ---
FT_BASE_VENCIDO      = 2.0     # Ft de un pedido recién vencido (vence hoy)
FT_MAX_VENCIDO       = 5.0     # Ft de un pedido en el tope de atraso (o más allá)
TOPE_DIAS_VENCIDO    = 28      # 4 semanas: a partir de aquí = máximamente urgente
DIAS_ATRASO_REVISION = 90      # atraso >= a esto se lista aparte para validar si sigue vivo

# --- Archivos de entrada (deben estar subidos a la sesión de Colab) ---
ARCHIVO_FOLIOS    = 'Plan de Corte por numero de parte_17.07.xlsx'   # plan de corte / nesteos del ERP
ARCHIVO_VENTAS    = 'Reporte de Cortos_17.07_Laser.xlsx'     # demanda (cortos)
ARCHIVO_INVENTARIO = 'Placa disponible_26.05_10.16.xlsx'     # opcional: si falta, se omite el filtro

# Una sola fecha de referencia para TODO el análisis (clave para la comparación justa)
hoy = datetime.now()

# %% [2] ----------------------------------------------------------------------
# CARGA Y LIMPIEZA (con validación de columnas para evitar corrupción silenciosa)
# -----------------------------------------------------------------------------
print("Cargando y limpiando datos...")
try:
    df_folios = pd.read_excel(ARCHIVO_FOLIOS)
    df_ventas = pd.read_excel(ARCHIVO_VENTAS)
except FileNotFoundError:
    print("Error: No se encontraron los archivos Excel obligatorios. Verifica las rutas/nombres.")
    sys.exit()

df_folios = df_folios.drop(0)
_cols_folios = ["No","Maquina","Folio","Programa","Material","Cantidad_placas","Surtido","proceso","Bloque",
                "Numero de parte","Cant. Piezas","Procesos","Corte","Setup","InicioCorte","Fin Corte","Acum",
                "Nesteo","Fecha","Disponible","Contenedor","terminadas"]
if len(df_folios.columns) != len(_cols_folios):
    raise ValueError(
        f"[Plan de Corte] Trae {len(df_folios.columns)} columnas y se esperaban {len(_cols_folios)}. "
        "El ERP cambió el formato del export: NO confíes en la secuencia hasta revisar el archivo."
    )
df_folios.columns = _cols_folios

df_ventas = df_ventas.drop(0)
_cols_ventas = ["OrdenEPS","Año","Sem","Bloque","Critico","Cliente","Ciudad","Producto Terminado",
                "Fecha","Real","Componente","Ing","Tipo","Transf","Proceso","Pzas","Hrs",
                "Nesteada","Cls","Placa","Esp","CodigoRadan","Color pintura","Peso","Familia",
                "Color","Comentario","Nesteos","idNesteos","TiempoCorteHrs","CorteLaser","Plasma","Nivelado","Pulidor","Limpieza","quintar filos",
                "Inspeccion","Etiqueta especial"]
if len(df_ventas.columns) != len(_cols_ventas):
    raise ValueError(
        f"[Reporte de Cortos] Trae {len(df_ventas.columns)} columnas y se esperaban {len(_cols_ventas)}. "
        "El ERP cambió el formato del export: NO confíes en el análisis hasta revisar el archivo."
    )
df_ventas.columns = _cols_ventas

df_folios['Numero de parte'] = df_folios['Numero de parte'].astype(str).str.strip()
df_folios['Cant. Piezas'] = pd.to_numeric(df_folios['Cant. Piezas'], errors='coerce').fillna(0)
df_folios['Corte'] = pd.to_numeric(df_folios['Corte'], errors='coerce').fillna(0) * 60  # horas -> minutos
df_ventas['Componente'] = df_ventas['Componente'].astype(str).str.strip()
df_ventas['Pzas'] = pd.to_numeric(df_ventas['Pzas'], errors='coerce')
df_ventas = df_ventas.dropna(subset=['Pzas'])

# Snapshot del plan CRUDO (antes del filtro de inventario) = plan Manual As-Is para la comparación
df_folios_manual = df_folios.copy()

# %% [3] ----------------------------------------------------------------------
# ADUANA DE INVENTARIO (opcional)
# -----------------------------------------------------------------------------
print("\nVerificando disponibilidad de Materia Prima en almacén...")
try:
    df_inventario = pd.read_excel(ARCHIVO_INVENTARIO)
    df_inventario.columns = df_inventario.columns.str.strip()
    df_inventario = df_inventario.rename(columns={'Placa': 'Material', 'Inventario': 'Stock_Almacen'})

    total_folios_original = len(df_folios)
    df_folios = pd.merge(df_folios, df_inventario[['Material', 'Stock_Almacen']], on='Material', how='left')
    df_folios['Stock_Almacen'] = df_folios['Stock_Almacen'].fillna(0)

    folios_sin_material = df_folios[df_folios['Stock_Almacen'] <= 0].copy()
    if not folios_sin_material.empty:
        print(f"⚠️ {len(folios_sin_material)} folios sin materia prima disponible (se excluyen del secuenciado).")
    df_folios = df_folios[df_folios['Stock_Almacen'] > 0].copy()
    print(f"Folios aprobados por almacén: {len(df_folios)} de {total_folios_original} originales.")
except FileNotFoundError:
    print("⚠️ No se encontró el archivo de Inventario. Se omite el filtro de materia prima.")

# %% [4] ----------------------------------------------------------------------
# DEMANDA URGENTE, CRUCE Y URGENCIA GRADUADA
# -----------------------------------------------------------------------------
df_ventas['Real'] = pd.to_datetime(df_ventas['Real'], errors='coerce')
DIAS_VISIBILIDAD = 7
fecha_limite = hoy + timedelta(days=DIAS_VISIBILIDAD)

df_ventas_urgentes = df_ventas[df_ventas['Real'] <= fecha_limite].copy()
df_ventas_agrupado = df_ventas_urgentes.groupby('Componente').agg({'Pzas': 'sum', 'Real': 'min'}).reset_index()

df_cruce = pd.merge(df_folios, df_ventas_agrupado, left_on='Numero de parte', right_on='Componente', how='left')
df_cruce['Pzas'] = df_cruce['Pzas'].fillna(0)
df_cruce['Real'] = pd.to_datetime(df_cruce['Real']).fillna(hoy + timedelta(days=100))
df_cruce['Dias_Faltantes'] = (df_cruce['Real'] - hoy).dt.days
df_cruce['Piezas_Utiles'] = df_cruce.apply(lambda x: min(x['Cant. Piezas'], x['Pzas']), axis=1)

# Urgencia graduada: el Ft de un vencido crece con los días de atraso hasta un tope.
_dias_venc = (-df_cruce['Dias_Faltantes']).clip(lower=0)
_pendiente = (FT_MAX_VENCIDO - FT_BASE_VENCIDO) / TOPE_DIAS_VENCIDO
_ft_vencido = FT_BASE_VENCIDO + _pendiente * _dias_venc.clip(upper=TOPE_DIAS_VENCIDO)
_ft_futuro = 1.0 / df_cruce['Dias_Faltantes'].where(df_cruce['Dias_Faltantes'] > 0, 1)
df_cruce['Ft_Parte'] = np.where(df_cruce['Dias_Faltantes'] <= 0, _ft_vencido, _ft_futuro)
df_cruce['Ft_Parte'] = np.where(df_cruce['Pzas'] == 0, 0, df_cruce['Ft_Parte'])

# Alerta de atraso extremo (posibles pedidos muertos) -> lista para revisión manual
folios_atraso_extremo = df_cruce[(df_cruce['Pzas'] > 0) &
                                 (df_cruce['Dias_Faltantes'] <= -DIAS_ATRASO_REVISION)].copy()
if not folios_atraso_extremo.empty:
    folios_atraso_extremo['Dias_Atraso'] = -folios_atraso_extremo['Dias_Faltantes']
    _rev = folios_atraso_extremo[['Folio', 'Maquina', 'Numero de parte', 'Dias_Atraso', 'Pzas']] \
        .sort_values('Dias_Atraso', ascending=False)
    print("\n" + "~"*60)
    print(f"REVISIÓN: {_rev['Folio'].nunique()} folios con atraso >= {DIAS_ATRASO_REVISION} días "
          f"(posibles pedidos muertos). Validar antes de confiar en su prioridad.")
    print("~"*60)
    print(_rev.head(10).to_string(index=False))

# %% [5] ----------------------------------------------------------------------
# SCORE POR FOLIO Y FILTRO
# -----------------------------------------------------------------------------
resumen_folios = df_cruce.groupby(['Folio', 'Maquina', 'Material', 'Corte']).agg(
    Total_Piezas_Folio=('Cant. Piezas', 'sum'),
    Total_Piezas_Utiles=('Piezas_Utiles', 'sum'),
    Ft_Maximo=('Ft_Parte', 'max')
).reset_index()

resumen_folios['Densidad_Fd'] = 0.0
_mask = resumen_folios['Total_Piezas_Folio'] != 0
resumen_folios.loc[_mask, 'Densidad_Fd'] = (
    resumen_folios.loc[_mask, 'Total_Piezas_Utiles'] / resumen_folios.loc[_mask, 'Total_Piezas_Folio']
).astype(np.float64)

resumen_folios['Score'] = (PESO_URGENCIA * resumen_folios['Ft_Maximo']) + (PESO_DENSIDAD * resumen_folios['Densidad_Fd'])
resumen_folios['Score'] = resumen_folios['Score'].replace(0, 0.001)
resumen_folios['Estatus'] = np.where(
    resumen_folios['Score'] >= UMBRAL_RECHAZO, 'Aprobado - Secuenciar', 'Rechazado - Re-Nesting'
)
vista_final = resumen_folios.sort_values(by='Score', ascending=False).reset_index(drop=True)
print(f"\nFolios aprobados para secuenciar: {(vista_final['Estatus']=='Aprobado - Secuenciar').sum()} de {len(vista_final)}")

# %% [6] ----------------------------------------------------------------------
# OPTIMIZADOR 1 — GUROBI (MILP exacto)  [idéntico a secuenciador_laser_v7.py]
# -----------------------------------------------------------------------------
def optimizar_gurobi(nombre_maquina, df_global, diagnostico=None, top_n=20, time_limit=300):
    # diagnostico: dict opcional que se rellena con MIPGap/Runtime/Status (experimento).
    # top_n / time_limit: tamaño de instancia y límite de Gurobi (para el benchmark de escalabilidad).
    df_maquina = df_global[(df_global['Estatus'] == 'Aprobado - Secuenciar') &
                           (df_global['Maquina'] == nombre_maquina)].copy()
    df_maquina = df_maquina.sort_values(by='Score', ascending=False).head(top_n)
    if df_maquina.empty:
        return pd.DataFrame()

    nodo_actual_df = pd.DataFrame({
        'Folio': ['Actual'], 'Maquina': [nombre_maquina],
        'Material': ['Ninguno'], 'Corte': [0.0], 'Score': [9999.0]
    })
    columnas_necesarias = ['Folio', 'Maquina', 'Material', 'Corte', 'Score']
    df_secuencia = pd.concat([nodo_actual_df, df_maquina[columnas_necesarias]], ignore_index=True)

    nodos = df_secuencia['Folio'].tolist()
    N = len(nodos)

    S = np.zeros((N, N))
    materiales = df_secuencia['Material'].tolist()
    for i in range(N):
        for j in range(N):
            if i != j and materiales[i] != materiales[j]:
                S[i][j] = 15.0

    T = {nodos[i]: df_secuencia['Corte'].iloc[i] for i in range(N)}
    P = {nodos[i]: df_secuencia['Score'].iloc[i] for i in range(N)}

    prob = pulp.LpProblem(f"Sec_{nombre_maquina.replace(' ', '_')}", pulp.LpMinimize)
    x = pulp.LpVariable.dicts(f"ruta_{nombre_maquina}", [(nodos[i], nodos[j]) for i in range(N) for j in range(N) if i != j], cat='Binary')
    C = pulp.LpVariable.dicts(f"C_{nombre_maquina}", nodos, lowBound=0, cat='Continuous')

    prob += pulp.lpSum(C[i] / P[i] for i in nodos if i != 'Actual') + 10 * pulp.lpSum(x[(nodos[i], nodos[j])] * S[i][j] for i in range(N) for j in range(N) if i != j)

    for i in nodos:
        prob += pulp.lpSum(x[(i, j)] for j in nodos if i != j) == 1
        prob += pulp.lpSum(x[(j, i)] for j in nodos if i != j) == 1

    M = sum(T.values()) + (15.0 * N)
    for i in range(N):
        for j in range(N):
            if i != j and nodos[j] != 'Actual':
                nodo_i, nodo_j = nodos[i], nodos[j]
                prob += C[nodo_j] >= C[nodo_i] + S[i][j] + T[nodo_j] - M * (1 - x[(nodo_i, nodo_j)])
    prob += C['Actual'] == 0

    prob.solve(pulp.GUROBI(timeLimit=time_limit, msg=False))

    if diagnostico is not None:
        _d = {'Maquina': nombre_maquina, 'Status': pulp.LpStatus[prob.status],
              'MIPGap': None, 'Runtime_s': None, 'Objetivo_solver': pulp.value(prob.objective)}
        try:  # atributos del modelo gurobipy subyacente (solo disponibles con Gurobi real)
            _d['MIPGap'] = float(prob.solverModel.MIPGap)
            _d['Runtime_s'] = float(prob.solverModel.Runtime)
        except Exception:
            pass
        diagnostico[nombre_maquina] = _d

    if pulp.value(prob.objective) is None:
        print(f"[!] {nombre_maquina}: Gurobi no encontró ruta en el tiempo límite.")
        return pd.DataFrame()

    secuencia_final = []
    nodo_actual = 'Actual'
    for _ in range(N):
        secuencia_final.append(nodo_actual)
        for j in nodos:
            if nodo_actual != j and pulp.value(x[(nodo_actual, j)]) > 0.5:
                nodo_actual = j
                break

    datos_resultado = []
    tiempo_acumulado = 0
    orden = 1
    for nodo in secuencia_final[1:]:
        idx = nodos.index(nodo)
        idx_prev = nodos.index(secuencia_final[secuencia_final.index(nodo)-1])
        setup = S[idx_prev][idx]
        tiempo_acumulado += setup
        inicio = tiempo_acumulado
        tiempo_acumulado += T[nodo]
        material = df_secuencia.loc[df_secuencia['Folio'] == nodo, 'Material'].values[0]
        score = P[nodo]
        datos_resultado.append({
            'Maquina': nombre_maquina, 'Orden': orden, 'Folio': nodo, 'Material': material,
            'Setup (min)': setup, 'Inicio (min)': inicio, 'Fin (min)': tiempo_acumulado,
            'Score_Interno': round(score, 2)
        })
        orden += 1
    return pd.DataFrame(datos_resultado)

# %% [7] ----------------------------------------------------------------------
# OPTIMIZADOR 2 — GENÉTICO (DEAP)  [idéntico a genetico_v1.py, con semilla estable]
# -----------------------------------------------------------------------------
def optimizar_genetico(nombre_maquina, df_global, semilla=None, top_n=20):
    # semilla: fija la semilla de la corrida (para variar entre corridas / benchmark).
    # top_n: tamaño de instancia (para el benchmark de escalabilidad).
    df_maquina = df_global[(df_global['Estatus'] == 'Aprobado - Secuenciar') &
                           (df_global['Maquina'] == nombre_maquina)].copy()
    df_maquina = df_maquina.sort_values(by='Score', ascending=False).head(top_n)
    if df_maquina.empty:
        return pd.DataFrame()

    nodo_actual_df = pd.DataFrame({'Folio': ['Actual'], 'Maquina': [nombre_maquina], 'Material': ['Ninguno'], 'Corte': [0.0], 'Score': [9999.0]})
    columnas_necesarias = ['Folio', 'Maquina', 'Material', 'Corte', 'Score']
    df_secuencia = pd.concat([nodo_actual_df, df_maquina[columnas_necesarias]], ignore_index=True)

    nodos = df_secuencia['Folio'].tolist()
    materiales = df_secuencia['Material'].tolist()
    T = df_secuencia['Corte'].tolist()
    P = df_secuencia['Score'].tolist()
    N = len(nodos)
    if N == 1:
        return pd.DataFrame()

    S = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i != j and materiales[i] != materiales[j]:
                S[i][j] = 15.0

    if N == 2:
        secuencia_final_indices = [0, 1]
    else:
        # Semilla estable por máquina, o la de la corrida si el experimento la inyecta.
        if semilla is None:
            semilla = int(''.join(ch for ch in nombre_maquina if ch.isdigit()) or "0")
        random.seed(semilla)
        np.random.seed(semilla)

        toolbox = base.Toolbox()
        toolbox.register("indices", random.sample, range(0, N-1), N-1)
        toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
        toolbox.register("population", tools.initRepeat, list, toolbox.individual)

        def eval_sequence(individual):
            z = 0.0
            tiempo_acumulado = 0.0
            total_setups = 0.0
            prev_idx = 0
            for gen in individual:
                idx = gen + 1
                setup = S[prev_idx][idx]
                tiempo_acumulado += setup + T[idx]
                z += tiempo_acumulado / P[idx]
                total_setups += setup
                prev_idx = idx
            z += 10.0 * total_setups
            return (z, )

        toolbox.register("evaluate", eval_sequence)
        toolbox.register("mate", tools.cxPartialyMatched)
        toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.1)
        toolbox.register("select", tools.selTournament, tournsize=3)

        pop = toolbox.population(n=200)
        hof = tools.HallOfFame(1)
        algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.2, ngen=300, halloffame=hof, verbose=False)
        mejor_individuo = hof[0]
        secuencia_final_indices = [0] + [gen + 1 for gen in mejor_individuo]

    datos_resultado = []
    tiempo_acumulado = 0
    orden = 1
    for i in range(1, len(secuencia_final_indices)):
        idx_prev = secuencia_final_indices[i-1]
        idx = secuencia_final_indices[i]
        nodo = nodos[idx]
        setup = S[idx_prev][idx]
        tiempo_acumulado += setup
        inicio = tiempo_acumulado
        tiempo_acumulado += T[idx]
        datos_resultado.append({
            'Maquina': nombre_maquina, 'Orden': orden, 'Folio': nodo, 'Material': materiales[idx],
            'Setup (min)': setup, 'Inicio (min)': inicio, 'Fin (min)': tiempo_acumulado,
            'Score_Interno': round(P[idx], 2)
        })
        orden += 1
    return pd.DataFrame(datos_resultado)

# %% [8] ----------------------------------------------------------------------
# EJECUCIÓN DE AMBOS OPTIMIZADORES (sobre los MISMOS datos)
# -----------------------------------------------------------------------------
def correr_optimizador(func, etiqueta):
    print(f"\n=== {etiqueta} ===")
    resultados = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=7) as executor:
        futuros = {executor.submit(func, maq, vista_final): maq for maq in LISTA_MAQUINAS}
        for futuro in concurrent.futures.as_completed(futuros):
            maquina = futuros[futuro]
            try:
                df_res = futuro.result()
                if not df_res.empty:
                    resultados.append(df_res)
                    print(f"[*] Secuencia generada para {maquina}")
                else:
                    print(f"[-] {maquina} omitida (sin folios aprobados o sin solución).")
            except Exception as e:
                print(f"[X] Error en {maquina}: {e}")
    if resultados:
        return pd.concat(resultados, ignore_index=True).sort_values(by=['Maquina', 'Orden']).reset_index(drop=True)
    return pd.DataFrame()

df_gurobi = correr_optimizador(optimizar_gurobi, "OPTIMIZACIÓN GUROBI (MILP)")
df_genetico = correr_optimizador(optimizar_genetico, "OPTIMIZACIÓN GENÉTICO (DEAP)")

# %% [9] ----------------------------------------------------------------------
# EVALUADOR EN MEMORIA (reloj + setups + cascada FIFO)  -> KPIs por escenario
# -----------------------------------------------------------------------------
def evaluar_plan(df_detalle, orden_cols, nombre, horas_turno=HORAS_TURNO):
    """
    df_detalle: filas a nivel (folio, parte) con columnas
        Maquina, Folio, Material, Corte (min), 'Numero de parte', Cant_piezas
    orden_cols: columnas que definen la SECUENCIA (p.ej. ['Maquina','Orden'] o ['Maquina','No']).
    Devuelve KPIs: Nivel de Servicio %, WIP, Horas de Setup, Piezas útiles surtidas.
    """
    if df_detalle.empty:
        return {'Escenario': nombre, 'Nivel_Servicio_%': 0.0, 'WIP_Piezas': 0, 'Horas_Setup': 0.0, 'Surtidas_Utiles': 0}

    minutos_limite = horas_turno * 60.0
    df = df_detalle.sort_values(orden_cols)

    resultados_sim = []
    total_setup_min = 0.0
    for maquina, grupo_maquina in df.groupby('Maquina', sort=False):
        reloj = 0.0
        material_previo = None
        for folio, grupo_folio in grupo_maquina.groupby('Folio', sort=False):
            row_folio = grupo_folio.iloc[0]
            material_actual = row_folio['Material']
            tiempo_corte = row_folio['Corte']
            setup = 15.0 if (material_previo is not None and material_actual != material_previo) else 0.0
            total_setup_min += setup
            inicio = reloj + setup
            fin = inicio + tiempo_corte
            for _, row_pieza in grupo_folio.iterrows():
                resultados_sim.append({
                    'Fin (min)': fin, 'Numero de parte': row_pieza['Numero de parte'], 'Piezas': row_pieza['Cant_piezas']
                })
            reloj = fin
            material_previo = material_actual

    df_sim = pd.DataFrame(resultados_sim)
    dentro = df_sim[df_sim['Fin (min)'] <= minutos_limite]
    produccion = dentro.groupby('Numero de parte')['Piezas'].sum().to_dict()

    total_surtidas = 0.0
    for _, row in df_ventas.sort_values(by=['Componente', 'Real']).iterrows():
        comp = row['Componente']
        if produccion.get(comp, 0) > 0:
            surtido = min(row['Pzas'], produccion[comp])
            produccion[comp] -= surtido
            total_surtidas += surtido

    solicitadas = df_ventas_urgentes['Pzas'].sum()
    cortadas = dentro['Piezas'].sum()
    wip = max(0, cortadas - total_surtidas)
    nivel_servicio = (total_surtidas / solicitadas * 100) if solicitadas > 0 else 0

    return {
        'Escenario': nombre,
        'Nivel_Servicio_%': round(nivel_servicio, 1),
        'WIP_Piezas': int(wip),
        'Horas_Setup': round(total_setup_min / 60, 2),
        'Surtidas_Utiles': int(total_surtidas)
    }

def construir_detalle_desde_secuencia(df_secuencia):
    """Adjunta el detalle por pieza (de df_folios) a una secuencia optimizada [Maquina, Orden, Folio]."""
    if df_secuencia.empty:
        return pd.DataFrame()
    det = df_folios.rename(columns={'Cant. Piezas': 'Cant_piezas'})[['Folio', 'Numero de parte', 'Cant_piezas', 'Corte', 'Material']]
    plan = df_secuencia[['Maquina', 'Orden', 'Folio']].merge(det, on='Folio', how='left')
    return plan

# %% [10] ---------------------------------------------------------------------
# CONSTRUIR LOS 3 ESCENARIOS Y EXTRAER KPIs
# -----------------------------------------------------------------------------
# Manual (As-Is): el plan crudo del ERP, en su orden original (Maquina, No)
plan_manual = df_folios_manual.rename(columns={'Cant. Piezas': 'Cant_piezas'})[
    ['Maquina', 'No', 'Folio', 'Material', 'Corte', 'Numero de parte', 'Cant_piezas']
].copy()
plan_manual['No'] = pd.to_numeric(plan_manual['No'], errors='coerce')

kpi_manual = evaluar_plan(plan_manual, ['Maquina', 'No'], 'Manual (As-Is)')
kpi_gurobi = evaluar_plan(construir_detalle_desde_secuencia(df_gurobi), ['Maquina', 'Orden'], 'Gurobi (MILP Exacto)')
kpi_genetico = evaluar_plan(construir_detalle_desde_secuencia(df_genetico), ['Maquina', 'Orden'], 'Genético (Metaheurística)')

kpis = [kpi_manual, kpi_gurobi, kpi_genetico]
print("\n" + "="*60)
print("KPIs POR ESCENARIO (turno de {:.0f} horas)".format(HORAS_TURNO))
print("="*60)
print(pd.DataFrame(kpis).to_string(index=False))

# %% [11] ---------------------------------------------------------------------
# DASHBOARD COMPARATIVO
# -----------------------------------------------------------------------------
def generar_dashboard_comparativo(kpis_lista):
    kpis_lista = [k for k in kpis_lista if k]
    if not kpis_lista:
        print("No hay escenarios válidos para comparar.")
        return
    df_kpis = pd.DataFrame(kpis_lista)
    colores = {'Manual (As-Is)': '#e74c3c', 'Gurobi (MILP Exacto)': '#00529b', 'Genético (Metaheurística)': '#27ae60'}

    fig = make_subplots(rows=1, cols=3, subplot_titles=(
        "Nivel de Servicio (%)<br><sup>Más alto es mejor</sup>",
        "Desperdicio por Setup (Horas)<br><sup>Más bajo es mejor</sup>",
        "Sobreproducción / WIP (Piezas)<br><sup>Más bajo es mejor</sup>"
    ))
    for _, row in df_kpis.iterrows():
        color = colores.get(row['Escenario'], '#888888')
        fig.add_trace(go.Bar(x=[row['Escenario']], y=[row['Nivel_Servicio_%']], marker_color=color, showlegend=False, text=f"{row['Nivel_Servicio_%']}%", textposition='auto'), row=1, col=1)
        fig.add_trace(go.Bar(x=[row['Escenario']], y=[row['Horas_Setup']], marker_color=color, showlegend=False, text=f"{row['Horas_Setup']}h", textposition='auto'), row=1, col=2)
        fig.add_trace(go.Bar(x=[row['Escenario']], y=[row['WIP_Piezas']], marker_color=color, showlegend=False, text=f"{row['WIP_Piezas']:,}", textposition='auto'), row=1, col=3)

    fig.update_layout(title_text=f"Dashboard de Validación: Comparativa de Algoritmos (Turno de {HORAS_TURNO:.0f} Horas)",
                      font=dict(size=14), template='plotly_white', height=500)
    fig.show()

generar_dashboard_comparativo(kpis)

# =============================================================================
# EXPERIMENTOS NIVEL 1 (para la tesis)
# =============================================================================
# Las celdas [8]-[11] dan una comparación RÁPIDA de KPIs operativos (una corrida).
# Para el capítulo de resultados de la tesis, las celdas de abajo agregan el rigor
# que un comité de posgrado espera:
#   - Comparación sobre la FUNCIÓN OBJETIVO del modelo (no solo KPIs derivados).
#   - Brecha de optimalidad de Gurobi (MIPGap): ¿el "exacto" probó ser óptimo?
#   - El genético corrido N veces: media, desviación, mejor y peor (es estocástico).
# Si solo quieres el experimento, corre [1]-[7] y salta directo a [12].

# %% [12] ---------------------------------------------------------------------
# FUNCIÓN OBJETIVO UNIFORME (la misma que minimizan Gurobi y el genético)
# -----------------------------------------------------------------------------
#   Z = Σ (tiempo_fin_folio / Score_folio) + 10 · Σ setups
# Se evalúa sobre el MISMO conjunto de folios (top-20 aprobados por máquina) para
# aislar la calidad del ORDEN. Un nodo ficticio 'Actual' (material 'Ninguno')
# arranca cada máquina, igual que en los optimizadores (por eso el 1er folio paga setup).
def calcular_objetivo_plan(df_orden, orden_cols):
    """Z total de la planta para una secuencia dada. df_orden: [Maquina, <orden>, Folio]."""
    if df_orden is None or df_orden.empty:
        return np.nan
    info = vista_final.drop_duplicates('Folio').set_index('Folio')[['Material', 'Corte', 'Score']]
    df = df_orden.sort_values(orden_cols)
    z_total = 0.0
    for maquina, grupo in df.groupby('Maquina', sort=False):
        prev_mat = 'Ninguno'   # nodo dummy 'Actual'
        t_acum = 0.0
        setups = 0.0
        for folio in grupo['Folio']:
            if folio not in info.index:
                continue
            mat = info.at[folio, 'Material']; corte = info.at[folio, 'Corte']; score = info.at[folio, 'Score']
            setup = 15.0 if mat != prev_mat else 0.0
            t_acum += setup + corte
            z_total += t_acum / score
            setups += setup
            prev_mat = mat
        z_total += 10.0 * setups
    return z_total

# Conjunto de candidatos = EXACTAMENTE lo que ven los optimizadores (top-20 aprobados/máquina)
df_candidatos = (vista_final[vista_final['Estatus'] == 'Aprobado - Secuenciar']
                 .sort_values('Score', ascending=False)
                 .groupby('Maquina', group_keys=False).head(20))

# Escenario Manual: esos mismos folios, pero en el orden original del ERP (columna 'No')
_orden_no = df_folios_manual.copy()
_orden_no['No'] = pd.to_numeric(_orden_no['No'], errors='coerce')
_orden_no = _orden_no.groupby('Folio')['No'].min()
plan_manual_obj = df_candidatos[['Maquina', 'Folio']].copy()
plan_manual_obj['No'] = plan_manual_obj['Folio'].map(_orden_no)
obj_manual = calcular_objetivo_plan(plan_manual_obj, ['Maquina', 'No'])
print(f"Objetivo Z — Manual (As-Is, mismo conjunto de folios): {obj_manual:,.1f}")

# %% [13] ---------------------------------------------------------------------
# GUROBI: 1 corrida (determinista) + brecha de optimalidad (MIPGap) y tiempo
# -----------------------------------------------------------------------------
import time
diag_gurobi = {}
res_g = []
t0 = time.perf_counter()
for maq in LISTA_MAQUINAS:
    df_res = optimizar_gurobi(maq, vista_final, diagnostico=diag_gurobi)
    if not df_res.empty:
        res_g.append(df_res)
tiempo_gurobi = time.perf_counter() - t0
df_gurobi_exp = (pd.concat(res_g, ignore_index=True).sort_values(['Maquina', 'Orden']).reset_index(drop=True)
                 if res_g else pd.DataFrame())

df_diag = pd.DataFrame(diag_gurobi.values())
gap_max = df_diag['MIPGap'].max() if (not df_diag.empty and df_diag['MIPGap'].notna().any()) else None
n_optimo = int((df_diag['Status'] == 'Optimal').sum()) if not df_diag.empty else 0
obj_gurobi = calcular_objetivo_plan(df_gurobi_exp, ['Maquina', 'Orden'])

print(f"\nGUROBI — objetivo Z: {obj_gurobi:,.1f} | tiempo total: {tiempo_gurobi:.1f}s")
print(f"Máquinas resueltas a óptimo comprobado: {n_optimo}/{len(df_diag)}"
      + (f" | MIPGap máx: {gap_max:.2%}" if gap_max is not None else " | (MIPGap no disponible)"))
print(df_diag.to_string(index=False))
# Chequeo de consistencia. El objetivo del solver incluye el "arco de cierre" del ciclo
# hamiltoniano (regreso al nodo 'Actual'): un setup de 15 min extra por máquina que NO es real.
# calcular_objetivo_plan usa la ruta ABIERTA (sin regreso), igual que la fitness del genético,
# por eso es el yardstick correcto y comparable. La diferencia esperada = 10·15 por máquina.
if not df_diag.empty and df_diag['Objetivo_solver'].notna().any():
    n_maq = int(df_diag['Objetivo_solver'].notna().sum())
    esperado = obj_gurobi + 10 * 15.0 * n_maq
    print(f"[check] Σ objetivo solver: {df_diag['Objetivo_solver'].sum():,.1f}  ==  "
          f"mi Z + arcos de cierre ({obj_gurobi:,.1f} + 10·15·{n_maq}) = {esperado:,.1f}")

# %% [14] ---------------------------------------------------------------------
# GENÉTICO: N corridas independientes (variación estocástica) + estadísticas
# -----------------------------------------------------------------------------
N_CORRIDAS_GENETICO = 30   # estándar en literatura de metaheurísticas (>= 30 para media estable)
registros = []
mejor_obj = np.inf
df_genetico_mejor = pd.DataFrame()

for r in range(N_CORRIDAS_GENETICO):
    res_r = []
    for maq in LISTA_MAQUINAS:
        base_maq = int(''.join(ch for ch in maq if ch.isdigit()) or "0")  # NO usar 'base' (pisa deap.base)
        df_res = optimizar_genetico(maq, vista_final, semilla=1000 * r + base_maq)  # semilla distinta por corrida
        if not df_res.empty:
            res_r.append(df_res)
    df_r = (pd.concat(res_r, ignore_index=True).sort_values(['Maquina', 'Orden']).reset_index(drop=True)
            if res_r else pd.DataFrame())
    obj_r = calcular_objetivo_plan(df_r, ['Maquina', 'Orden'])
    kpi_r = evaluar_plan(construir_detalle_desde_secuencia(df_r), ['Maquina', 'Orden'], 'Genético')
    registros.append({'Corrida': r + 1, 'Objetivo': obj_r,
                      'Nivel_Servicio_%': kpi_r['Nivel_Servicio_%'], 'WIP_Piezas': kpi_r['WIP_Piezas'],
                      'Horas_Setup': kpi_r['Horas_Setup'], 'Surtidas_Utiles': kpi_r['Surtidas_Utiles']})
    if obj_r < mejor_obj:
        mejor_obj = obj_r
        df_genetico_mejor = df_r

df_corridas = pd.DataFrame(registros)
print(f"\nGENÉTICO — {N_CORRIDAS_GENETICO} corridas. Estadísticas:")
print(df_corridas[['Objetivo', 'Nivel_Servicio_%', 'WIP_Piezas', 'Horas_Setup']]
      .describe().loc[['mean', 'std', 'min', 'max']].round(2).to_string())

# %% [15] ---------------------------------------------------------------------
# TABLA COMPARATIVA FINAL Y EXPORTACIÓN (capítulo de resultados)
# -----------------------------------------------------------------------------
def _brecha(z):
    return round((z - obj_gurobi) / obj_gurobi * 100, 1) if obj_gurobi else np.nan

tabla = pd.DataFrame([
    {'Método': 'Manual (As-Is)',   'Objetivo_Z': round(obj_manual, 1),                 'Objetivo_Z_std': np.nan,                           'Brecha_vs_Gurobi_%': _brecha(obj_manual),                 'Corridas': 1},
    {'Método': 'Gurobi (MILP)',    'Objetivo_Z': round(obj_gurobi, 1),                 'Objetivo_Z_std': np.nan,                           'Brecha_vs_Gurobi_%': 0.0,                                 'Corridas': 1},
    {'Método': 'Genético (media)', 'Objetivo_Z': round(df_corridas['Objetivo'].mean(), 1), 'Objetivo_Z_std': round(df_corridas['Objetivo'].std(), 1), 'Brecha_vs_Gurobi_%': _brecha(df_corridas['Objetivo'].mean()), 'Corridas': N_CORRIDAS_GENETICO},
    {'Método': 'Genético (mejor)', 'Objetivo_Z': round(df_corridas['Objetivo'].min(), 1),  'Objetivo_Z_std': np.nan,                           'Brecha_vs_Gurobi_%': _brecha(df_corridas['Objetivo'].min()),  'Corridas': N_CORRIDAS_GENETICO},
])
print("\n" + "=" * 64)
print("TABLA COMPARATIVA — Objetivo Z (menor = mejor)")
print("=" * 64)
print(tabla.to_string(index=False))
if gap_max is not None:
    print(f"\nGurobi: {n_optimo}/{len(df_diag)} máquinas a óptimo comprobado (MIPGap máx {gap_max:.2%}).")

sello = hoy.strftime('%Y%m%d_%H%M')
with pd.ExcelWriter(f"Experimento_Nivel1_{sello}.xlsx", engine='openpyxl') as w:
    tabla.to_excel(w, sheet_name='Resumen', index=False)
    df_corridas.to_excel(w, sheet_name='Corridas_Genetico', index=False)
    df_diag.to_excel(w, sheet_name='Diagnostico_Gurobi', index=False)
print(f"\nResultados exportados a: Experimento_Nivel1_{sello}.xlsx")

# NOTA — comparación en varios días (recomendado para la tesis):
# Vuelve a correr el notebook cambiando ARCHIVO_FOLIOS/ARCHIVO_VENTAS por otro día.
# Cada corrida genera su propio Experimento_Nivel1_*.xlsx; apílalos para mostrar que
# el patrón (Gurobi <= Genético < Manual) se sostiene entre datasets, no en uno solo.


# =============================================================================
# EXPERIMENTOS NIVEL 2 — PRUEBA DE ESCALABILIDAD (el argumento central de la tesis)
# =============================================================================
# Barre el tamaño de instancia N (folios en UN problema de secuenciación) y mide
# cómo escala cada motor. La instancia se arma tomando los N folios aprobados de
# mayor Score (de todas las máquinas, agrupados en una sola instancia 'BENCH'):
# esto aísla la variable de interés —el TAMAÑO— usando datos reales, y permite
# llegar a N grandes que una sola máquina no alcanzaría. Es un BENCHMARK de los
# algoritmos, no un plan de producción.
#
# Hipótesis a demostrar: el tiempo de Gurobi (exacto) crece de forma explosiva y
# deja de cerrar el gap más allá de cierto N, mientras el genético crece suave y
# se mantiene cerca del óptimo — justificando la metaheurística.

# %% [16] ---------------------------------------------------------------------
# PARÁMETROS DEL BENCHMARK
# -----------------------------------------------------------------------------
MODO_RAPIDO = True   # True: barrido corto para probar. False: barrido completo para la tesis.
if MODO_RAPIDO:
    TAMANOS_N = [8, 10, 12, 15, 20]
    K_CORRIDAS_GEN = 5
    TIME_LIMIT_BENCH = 60
else:
    TAMANOS_N = [8, 10, 12, 15, 20, 25, 30, 40, 50]
    K_CORRIDAS_GEN = 10
    TIME_LIMIT_BENCH = 120

# Pool de folios aprobados (todas las máquinas) para construir instancias de tamaño controlado
_pool = (vista_final[vista_final['Estatus'] == 'Aprobado - Secuenciar']
         .sort_values('Score', ascending=False).reset_index(drop=True))
print(f"Pool de folios aprobados: {len(_pool)}  (N máximo tratable en el benchmark)")

# Mapa Folio -> No original del ERP (para el escenario Manual de cada instancia)
_tmp = df_folios_manual.copy()
_tmp['No'] = pd.to_numeric(_tmp['No'], errors='coerce')
_no_por_folio = _tmp.groupby('Folio')['No'].min()

def instancia_bench(N):
    """N folios de mayor Score, etiquetados como una sola 'máquina' (una instancia de tamaño N)."""
    inst = _pool.head(N).copy()
    inst['Maquina'] = 'BENCH'
    return inst

# %% [17] ---------------------------------------------------------------------
# BARRIDO: para cada N, correr Manual / Gurobi / Genético y medir tiempo y calidad
# -----------------------------------------------------------------------------
import time
filas_bench = []
for N in TAMANOS_N:
    inst = instancia_bench(N)
    n_real = len(inst)
    if n_real < N:
        print(f"(N={N}: el pool solo tiene {n_real} folios; se detiene el barrido aquí)")

    # Manual: la instancia en el orden original del ERP
    man = inst[['Maquina', 'Folio']].copy()
    man['No'] = man['Folio'].map(_no_por_folio)
    z_man = calcular_objetivo_plan(man, ['Maquina', 'No'])

    # Gurobi (1 corrida, con su gap y tiempo)
    diag = {}
    t0 = time.perf_counter()
    seq_g = optimizar_gurobi('BENCH', inst, diagnostico=diag, top_n=n_real, time_limit=TIME_LIMIT_BENCH)
    t_g = time.perf_counter() - t0
    z_g = calcular_objetivo_plan(seq_g, ['Maquina', 'Orden'])
    dg = diag.get('BENCH', {})
    gap_g, status_g = dg.get('MIPGap'), dg.get('Status')

    # Genético (K corridas: mejor Z y tiempo promedio)
    z_runs, t_runs = [], []
    for k in range(K_CORRIDAS_GEN):
        t0 = time.perf_counter()
        seq = optimizar_genetico('BENCH', inst, semilla=k, top_n=n_real)
        t_runs.append(time.perf_counter() - t0)
        z_runs.append(calcular_objetivo_plan(seq, ['Maquina', 'Orden']))
    z_gen_best, z_gen_mean, t_gen = np.nanmin(z_runs), np.nanmean(z_runs), float(np.mean(t_runs))

    filas_bench.append({
        'N': n_real, 'Z_Manual': round(z_man, 1),
        'Z_Gurobi': round(z_g, 1), 'Gurobi_s': round(t_g, 2), 'Gurobi_MIPGap': gap_g, 'Gurobi_Status': status_g,
        'Z_Gen_mejor': round(z_gen_best, 1), 'Z_Gen_media': round(z_gen_mean, 1), 'Genetico_s': round(t_gen, 3),
    })
    _gs = 'n/d' if gap_g is None else f'{gap_g:.1%}'
    print(f"N={n_real:>3} | Gurobi {t_g:>7.1f}s (gap {_gs}) Z={z_g:>8,.0f} | "
          f"Gen {t_gen:>5.2f}s Z={z_gen_best:>8,.0f} | Manual Z={z_man:>8,.0f}")
    if n_real < N:
        break

df_bench = pd.DataFrame(filas_bench)
print("\n" + df_bench.to_string(index=False))

# %% [18] ---------------------------------------------------------------------
# GRÁFICAS DE ESCALABILIDAD
# -----------------------------------------------------------------------------
fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "Tiempo de cómputo vs Tamaño (eje log)<br><sup>Gurobi explota, Genético crece suave</sup>",
    "Calidad de la solución (Objetivo Z)<br><sup>Menor es mejor</sup>",
    "Brecha de optimalidad de Gurobi<br><sup>% sin cerrar al límite de tiempo</sup>"))

# 1) Tiempo (log)
fig.add_trace(go.Scatter(x=df_bench['N'], y=df_bench['Gurobi_s'], name='Gurobi', mode='lines+markers', marker_color='#00529b'), 1, 1)
fig.add_trace(go.Scatter(x=df_bench['N'], y=df_bench['Genetico_s'], name='Genético', mode='lines+markers', marker_color='#27ae60'), 1, 1)
fig.update_yaxes(type='log', title_text='segundos (log)', row=1, col=1)
fig.update_xaxes(title_text='N (folios en la instancia)', row=1, col=1)

# 2) Calidad Z
fig.add_trace(go.Scatter(x=df_bench['N'], y=df_bench['Z_Manual'], name='Manual', mode='lines+markers', marker_color='#e74c3c', showlegend=False), 1, 2)
fig.add_trace(go.Scatter(x=df_bench['N'], y=df_bench['Z_Gurobi'], name='Gurobi', mode='lines+markers', marker_color='#00529b', showlegend=False), 1, 2)
fig.add_trace(go.Scatter(x=df_bench['N'], y=df_bench['Z_Gen_mejor'], name='Genético', mode='lines+markers', marker_color='#27ae60', showlegend=False), 1, 2)
fig.update_xaxes(title_text='N', row=1, col=2)

# 3) MIPGap de Gurobi
_gap_pct = pd.to_numeric(df_bench['Gurobi_MIPGap'], errors='coerce') * 100
fig.add_trace(go.Bar(x=df_bench['N'], y=_gap_pct, marker_color='#00529b', showlegend=False), 1, 3)
fig.update_xaxes(title_text='N', row=1, col=3)
fig.update_yaxes(title_text='MIPGap %', row=1, col=3)

fig.update_layout(title_text="Escalabilidad: Gurobi (exacto) vs Genético (metaheurística)",
                  template='plotly_white', height=460, legend=dict(orientation="h", y=1.15, x=0.0))
fig.show()

# %% [19] ---------------------------------------------------------------------
# EXPORTACIÓN DEL BENCHMARK
# -----------------------------------------------------------------------------
_sello = hoy.strftime('%Y%m%d_%H%M')
df_bench.to_excel(f"Benchmark_Escalabilidad_{_sello}.xlsx", index=False)
print(f"\nBenchmark exportado a: Benchmark_Escalabilidad_{_sello}.xlsx")
print("Lectura para la tesis: identifica el N donde Gurobi deja de cerrar el gap (MIPGap > 0) "
      "y compara su tiempo contra el del genético a ese mismo N.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 64.4 MB/s eta 0:00:00
Cargando y limpiando datos...

Verificando disponibilidad de Materia Prima en almacén...
⚠️ No se encontró el archivo de Inventario. Se omite el filtro de materia prima.

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
REVISIÓN: 9 folios con atraso >= 90 días (posibles pedidos muertos). Validar antes de confiar en su prioridad.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
   Folio  Maquina Numero de parte  Dias_Atraso  Pzas
374120.0 LASER 02     91756376-RA          124 123.0
374121.0 LASER 02     91756376-RA          124 123.0
374125.0 LASER 02     90417394-RB          124  48.0
374125.0 LASER 02     91756368-RA          124  83.0
374125.0 LASER 02     91756370-RA          124  62.0
374125.0 LASER 02     91754121-RA      

Objetivo Z — Manual (As-Is, mismo conjunto de folios): 10,503.6

GUROBI — objetivo Z: 7,759.5 | tiempo total: 1802.9s
Máquinas resueltas a óptimo comprobado: 1/7 | MIPGap máx: 35.36%
 Maquina     Status   MIPGap  Runtime_s  Objetivo_solver
LASER 01 Not Solved 0.138769 300.000544      1170.574311
LASER 02 Not Solved 0.353610 300.001279       847.975445
LASER 03 Not Solved 0.012856 300.000540      1986.861756
LASER 04    Optimal 0.000000   0.035370       605.400000
LASER 05 Not Solved 0.020231 300.000780      1858.490573
LASER 06 Not Solved 0.164420 300.000522      1111.775212
LASER 08 Not Solved 0.336215 300.000498      1228.379462
[check] Σ objetivo solver: 8,809.5  ==  mi Z + arcos de cierre (7,759.5 + 10·15·7) = 8,809.5

GENÉTICO — 30 corridas. Estadísticas:
      Objetivo  Nivel_Servicio_%  WIP_Piezas  Horas_Setup
mean   8023.90              3.17      371.73        10.09
std     133.31              0.37      131.78         0.21
min    7779.65              2.40      197.00         9.


Benchmark exportado a: Benchmark_Escalabilidad_20260717_1422.xlsx
Lectura para la tesis: identifica el N donde Gurobi deja de cerrar el gap (MIPGap > 0) y compara su tiempo contra el del genético a ese mismo N.
